# SignGuard: reproducible Colab runner

Trains and evaluates every model in the project on a free Colab GPU. Run the cells from top to bottom.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

| Section | What it does | Approx. time on a T4 |
|---|---|---|
| 1-3 | Get the code, install packages, download GTSRB | 5 min |
| 4 | Data check and unit tests | 2 min |
| 5 | Train ResNet-18, clean vs weather-augmented (core experiment) | 8 min |
| 6 | Train SmallCNN and ConvNeXt-tiny (comparison) | 25-30 min |
| 7 | Evaluate all models, make figures | 15 min |
| 8 | Back up to Google Drive, launch the app | 5 min |

## 1. Get the code
Set `FOLDER` to the project folder name inside the course repository.

In [ ]:
FOLDER = "<your-folder>"          # e.g. "abc123": the folder that contains app.py
!git clone -q https://github.com/xahiru/PT22026.git
import os
os.chdir(f"PT22026/{FOLDER}")
print(os.getcwd())
!ls


*Alternative without GitHub:* upload `signguard.zip` with `from google.colab import files; files.upload()`, then `!unzip -q -o signguard.zip` and `%cd signguard`.

## 2. Install packages

In [ ]:
!pip install -q gradio matplotlib pytest kagglehub

## 3. Download the dataset
GTSRB (Stallkamp et al., 2012) from Kaggle's public copy. It is not stored in the repository.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
!rm -rf /content/gtsrb
!ln -s "{path}" /content/gtsrb       # the later steps use this fixed name
!ls /content/gtsrb

## 4. Check the data and run the tests
Expected: 39,209 training images, 12,630 test images, 43 classes; the leakage-free split gives 32,610 training and 6,599 validation images (220 distinct validation signs).

In [ ]:
import sys; sys.path.insert(0, '.')
from src.data import find_root, load_index, group_split
root = find_root('/content/gtsrb')
train, test = load_index(root, 'train'), load_index(root, 'test')
tr, va = group_split(train)
print('train images:', len(train), '| test images:', len(test), '| classes:', train.label.nunique())
print('after split -> train:', len(tr), '| validation:', len(va), '| distinct validation signs:', va.group.nunique())

In [ ]:
!python -m pytest -q -k 'not pipeline'

## 5. Core experiment: ResNet-18, clean vs weather-augmented
*Clean*: normal images only. *Weather*: half of the training images get random fog, night, rain, blur or noise (severity 1-4). Glare and severity 5 are never seen in training.

In [ ]:
!python training/train.py --data-root /content/gtsrb --model resnet18 --epochs 8 --weather-p 0 --name resnet18_clean

In [ ]:
!python training/train.py --data-root /content/gtsrb --model resnet18 --epochs 8 --weather-p 0.5 --name resnet18_weather

## 6. Comparison models: our own SmallCNN and ConvNeXt-tiny

In [ ]:
!python training/train.py --data-root /content/gtsrb --model smallcnn --no-pretrained --epochs 15 --weather-p 0 --name smallcnn_clean
!python training/train.py --data-root /content/gtsrb --model smallcnn --no-pretrained --epochs 15 --weather-p 0.5 --name smallcnn_weather

In [ ]:
!python training/train.py --data-root /content/gtsrb --model convnext_tiny --epochs 6 --weather-p 0 --name convnext_tiny_clean
!python training/train.py --data-root /content/gtsrb --model convnext_tiny --epochs 6 --weather-p 0.5 --name convnext_tiny_weather

## 7. Evaluate and plot
Fits the temperature on the clean validation set, tests every weather condition at severities 1, 3 and 5, and writes `evaluation/results/` and `evaluation/figures/`.

In [ ]:
RUNS = ["resnet18_clean", "resnet18_weather", "smallcnn_clean", "smallcnn_weather",
        "convnext_tiny_clean", "convnext_tiny_weather"]
ckpts = " ".join(f"outputs/{r}/model.pt" for r in RUNS)
!python evaluation/evaluate.py --data-root /content/gtsrb --ckpt {ckpts}
!python evaluation/make_plots.py

In [ ]:
import pandas as pd
from IPython.display import Image, display
df = pd.read_csv("evaluation/results/robustness.csv")
t = df[df.severity.isin([0, 3, 5])].pivot_table(index="run", columns=["condition", "severity"], values="accuracy")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print((t * 100).round(1).to_string())
for f in ["robustness", "reliability", "coverage", "training_curves"]:
    display(Image(f"evaluation/figures/{f}.png"))

## 8. Back up and launch the app
Model weights are not stored in the repository. Back them up to Google Drive (share the folder by link and put the link in the README).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/signguard_backup
!cp -r outputs evaluation /content/drive/MyDrive/signguard_backup/
!ls /content/drive/MyDrive/signguard_backup

In [ ]:
import os
os.environ['SIGNGUARD_CKPTS'] = ','.join([
    'ResNet18 clean=outputs/resnet18_clean/model.pt',
    'ResNet18 weather=outputs/resnet18_weather/model.pt',
    'SmallCNN clean=outputs/smallcnn_clean/model.pt',
    'SmallCNN weather=outputs/smallcnn_weather/model.pt',
    'ConvNeXt clean=outputs/convnext_tiny_clean/model.pt',
    'ConvNeXt weather=outputs/convnext_tiny_weather/model.pt'])
from app import demo
demo.launch(share=True)        # open the gradio.live link